# Trung bình 4 lớp ẩn cuối — PhoBERT, bài category (nhúng pooled, train + dev)

Thay vector lớp cuối bằng trung bình 4 lớp ẩn cuối (`output_hidden_states=True`) rồi mean pooling
có mặt nạ như cũ. Shape đầu ra `[n, 768]`; `max_len=256`; chỉ nhúng `train` và `dev` — **không đụng `test`** (Rule 5).
PhoBERT cần tách từ (`underthesea`) trước khi nhúng; chạy trên T4 ước tính ≈ 35–40 phút (chưa đo).

Mốc để so (dev, lớp cuối, 5 seed): CE 0,6032 ± 0,0029; cw effective β=0,999 + ls 0,05 0,6071 ± 0,0037.

**Trước khi chạy:** Runtime → Change runtime type → **GPU**. Cần trên Drive của tài khoản này:
`MyDrive/vietjobs_splits/` (train.csv dev.csv test.csv manifest.json) và `MyDrive/vietjobs_src_last4.zip` (mã có `--layers`).
Notebook này **chỉ nhúng**; huấn luyện Dense chạy trên Mac, `--device cpu` (Rule 13 mục 5).

In [ ]:
!nvidia-smi | head -15
!df -h /content | tail -1

In [ ]:
import os
if not os.path.isdir('vietjobs'):
    !git clone -q -b t8-rnn-colab https://github.com/boy2407/vietjobs.git
%cd vietjobs
# KHÔNG cài lại torch — giữ bản CUDA sẵn có của Colab.
!pip install -q "transformers>=4.38,<4.47" "pyarrow>=15.0" "scikit-learn>=1.5" \
    "scipy>=1.13" "joblib>=1.4" "underthesea>=6.8" sentencepiece protobuf pytest
!pip install -q -e . --no-deps
%env PYTHONPATH=src

## Chép mã có `--layers` đè lên bản clone

Nhánh trên GitHub chưa có `--layers`. Giải nén `vietjobs_src_last4.zip` (thư mục `src/vietjobs`) từ Drive đè lên.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -oq /content/drive/MyDrive/vietjobs_src_last4.zip -d .
assert '--layers' in open('src/vietjobs/dl/encode.py').read(), 'encode.py chưa có --layers'
print('encode.py có --layers')

## Khôi phục splits

In [ ]:
!python notebooks/colab_setup.py --family raw --no-train-token --skip-tests

In [ ]:
!python -m pytest -q tests/test_dl_encode.py

## Nhúng `dev` trước, rồi `train`

Hai lệnh riêng: cache pooled chỉ được ghi khi xong cả split, nên bị ngắt giữa `train` chỉ mất lượt `train`.
Hết VRAM thì giảm `--batch` (64 → 32).

In [ ]:
!python -m vietjobs.dl.encode --task category --splits dev --encoder phobert --max-len 256 --layers 4 --device cuda --batch 64
!cat artifacts/embeddings/*/dev-raw-len256-last4.json

In [ ]:
!python -m vietjobs.dl.encode --task category --splits train --encoder phobert --max-len 256 --layers 4 --device cuda --batch 64

## Kiểm shape và sao lưu lên Drive

In [ ]:
import glob, numpy as np, json
for split, n in (('train', 34354), ('dev', 3812)):
    p = glob.glob(f'artifacts/embeddings/*/{split}-raw-len256-last4.npy')[0]
    a = np.load(p, mmap_mode='r')
    assert a.shape == (n, 768), (p, a.shape)
    assert np.isfinite(a).all()
    print(p, a.shape, a.dtype, json.load(open(p.replace('.npy', '.json')))['layers'], 'lớp')

In [ ]:
!mkdir -p /content/drive/MyDrive/vietjobs_cache_last4
!cp artifacts/embeddings/*/*-raw-len256-last4.* /content/drive/MyDrive/vietjobs_cache_last4/
!ls -l /content/drive/MyDrive/vietjobs_cache_last4/
!md5sum /content/drive/MyDrive/vietjobs_cache_last4/*-last4.npy

## Mang về máy chính

Tải 4 tệp trong `MyDrive/vietjobs_cache_last4/` (`train-raw-len256-last4.{npy,json}`, `dev-raw-len256-last4.{npy,json}`),
đặt vào `artifacts/embeddings/<SPLIT_TAG>/` trên Mac, so `md5sum`, rồi huấn luyện:

```bash
PYTHONPATH=src .venv-dl/bin/python -m vietjobs.dl.train_dl --task category --encoder phobert --max-len 256 --layers 4 --seed 42 --run-id dl-cat-pho-last4-ce-s42
```